# COMP4318/5318 Assignment 2: Image Classification

This template notebook includes code to load the  dataset and a skeleton for the main sections that should be included in the notebook. Please stick to this struture for your submitted notebook.

Please focus on making your code clear, with appropriate variable names and whitespace. Include comments and markdown text to aid the readability of your code where relevant. See the specification and marking criteria in the associated specification to guide you when completing your implementation.

## Setup and dependencies
Please use this section to list and set up all your required libraries/dependencies and your plotting environment. 

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
import warnings

warnings.filterwarnings("ignore")

# Global Theme Settings
sns.set_theme(style="whitegrid", rc={
    "axes.facecolor": "white",
    "axes.edgecolor": "#003366",
    "axes.labelcolor": "#003366",
    "xtick.color": "#003366",
    "ytick.color": "#003366",
    "text.color": "#003366"
})

# Define your own colors
deep_blue = "#003366"
deep_red = "#990000"

## 1. Data loading, exploration, and preprocessing


Code to load the dataset is provided in the following cell. Please proceed with your data exploration and preprocessing in the remainder of this section.

In [ ]:
X_train = np.load('X_train.npy')
X_test = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test = np.load('y_test.npy')

y_train_raw = y_train.reshape(-1).astype(int)
y_test_raw  = y_test.reshape(-1).astype(int)

print(f"Train dataset: X={X_train.shape}, y={y_train_raw.shape}")
print(f"Test dataset: X={X_test.shape},  y={y_test_raw.shape}")

class_names = [
    'airplane','automobile','bird','cat','deer',
    'dog','frog','horse','ship','truck'
]

In [ ]:
# -------------------------------
# 2. EDA
# -------------------------------
# (1) Category distribution
class_counts = pd.Series(y_train_raw).value_counts().sort_index()
plt.figure(figsize=(8,4))
sns.barplot(
    x=class_names,
    y=class_counts.values,
    hue=class_names, legend=False,
    palette=[deep_blue if i%2==0 else deep_red for i in range(len(class_names))]
)
plt.title("Distribution of the number of samples in each category of the CIFAR-10 training set", fontsize=12, color=deep_red, weight='bold')
plt.ylabel("Number of samples", color=deep_blue)
plt.xticks(rotation=45, color=deep_blue)
plt.tight_layout()
plt.show()


# (2) Comparison of RGB channel means
r_mean = X_train[:,:,:,0].mean()
g_mean = X_train[:,:,:,1].mean()
b_mean = X_train[:,:,:,2].mean()
print(f"RGB channel mean: R={r_mean:.2f}, G={g_mean:.2f}, B={b_mean:.2f}")

# (3) Overall pixel distribution
plt.hist(X_train.ravel(), bins=50, color="#003366", alpha=0.8)
plt.title("Pixel intensity distribution (0–255)", color="#990000")
plt.xlabel("Pixel value"); plt.ylabel("Frequency"); plt.show()

# (4) Random sample display for each category
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    idx = np.where(y_train_raw==i)[0][0]
    plt.imshow(X_train[idx].astype(np.uint8))
    plt.axis('off')
    plt.title(class_names[i], fontsize=9)
plt.suptitle("Example images of each category", color="#990000")
plt.tight_layout(); plt.show()

# (5) Image complexity (standard deviation distribution)
variances = [np.std(img) for img in X_train[:5000]]  # 采样5000张
sns.histplot(variances, color="#003366", bins=40)
plt.title("Standard deviation distribution of image pixel intensities (complexity index)", color="#990000")
plt.xlabel("Standard deviation"); plt.ylabel("Frequency"); plt.show()

### Examples of preprocessed data
Please print/display some examples of your preprocessed data here.

In [ ]:
import numpy as np
import os
import random
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers

# =========================================
# Step 0. Global Configuration
# =========================================
np.random.seed(0)
random.seed(0)
os.environ["OMP_NUM_THREADS"] = "1"

# Reading raw data
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_test  = np.load("X_test.npy")
y_test  = np.load("y_test.npy")

print(f"Loaded data: X_train={X_train.shape}, X_test={X_test.shape}")

# =========================================================
# Approach 1️: impute + standardize
# =========================================================
print("\n [Path 1] Impute missing values + Standardize")

# Flatten
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Missing value filling
imputer = SimpleImputer(strategy="mean")
X_train_imp = imputer.fit_transform(X_train_flat)
X_test_imp = imputer.transform(X_test_flat)

# standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# save
np.save("X_train_preprocessed.npy", X_train_scaled)
np.save("X_test_preprocessed.npy", X_test_scaled)
np.save("y_train_preprocessed.npy", y_train)
np.save("y_test_preprocessed.npy", y_test)

print("Saved: impute_standardize version")

# =========================================================
# Path 2️: normalize + flatten + onehot + val split
# =========================================================
print("\n [Path 2] Normalize + Flatten + One-Hot + Validation split")

#Normalization
X_train_f = X_train.astype("float32") / 255.0
X_test_f = X_test.astype("float32") / 255.0

# Flatten
X_train_vec = X_train_f.reshape(X_train_f.shape[0], -1)
X_test_vec = X_test_f.reshape(X_test_f.shape[0], -1)

# One-hot coding
y_train = y_train.reshape(-1)
y_test = y_test.reshape(-1)
try:
    enc = OneHotEncoder(sparse_output=False)
except TypeError:
    enc = OneHotEncoder(sparse=False)

y_train_oh = enc.fit_transform(y_train[:, None])
y_test_oh = enc.transform(y_test[:, None])

# Training/validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_vec, y_train_oh, test_size=0.2, random_state=0, stratify=y_train
)

# save
np.savez_compressed(
    "cifar10_mlp_preprocessed.npz",
    X_tr=X_tr, y_tr=y_tr,
    X_val=X_val, y_val=y_val,
    X_test=X_test_vec, y_test=y_test_oh,
)
print("Saved: normalize_onehot version (cifar10_mlp_preprocessed.npz)")

# =========================================================
# Path 3️: keras Normalization layer (for CNN)
# =========================================================
print("\n🔹 [Path 3] Keras Normalization Layer (for CNNs)")

# 1 Normalize to [0,1]
X_train = X_train.astype("float32")
X_test  = X_test.astype("float32")
if X_train.max() > 1.0:
    X_train /= 255.0
    X_test  /= 255.0

# 2 Standardization — compute statistics using training set only
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train)  # calculate mean and variance only on training set

# Load the raw image (100 here is just an example)
X_train_raw = np.load('Assignment2Data/X_train.npy')

# 3 Select the 100th image for demonstration
idx = 100
original_img   = X_train_raw[idx]  # Original unnormalized image (uint8, 0~255)
normalized_img = X_train[idx]      # Normalized image (float32, 0~1)

# 4 Standardize (Normalization layer expects shape (1, 32, 32, 3))
img_norm_batched = normalizer(normalized_img[None, ...])  # explicitly add batch dimension
img_norm = img_norm_batched[0].numpy()                    # remove batch dimension -> (32, 32, 3)

# 5 For visualization, scale the standardized result to [0,1]
eps = 1e-8
img_norm_disp = (img_norm - img_norm.min()) / (img_norm.max() - img_norm.min() + eps)

# 6 Plot original vs normalized image
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(original_img.astype("uint8"))
plt.title("Original (uint8 0~255)")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(img_norm_disp)
plt.title("Normalized & Standardized")
plt.axis("off")

plt.show()

## 2. Algorithm design and setup

### Algorithm of choice from first six weeks of course

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report

# ===============================
# Step 1. Load the "preprocessed" data
# ===============================
print(" Loading preprocessed data...")

X_train = np.load('X_train_preprocessed.npy').astype(np.float16)
X_test = np.load('X_test_preprocessed.npy').astype(np.float16)
y_train = np.load('y_train_preprocessed.npy')
y_test = np.load('y_test_preprocessed.npy')

print(f"X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"X_test shape: {X_test.shape}, dtype: {X_test.dtype}")

# ===============================
# Step 2. Sampling to save memory 
# ===============================
sample_ratio = 0.25   # Use 25% of the samples to adjust the parameters
n_samples = int(X_train.shape[0] * sample_ratio)
X_train_small = X_train[:n_samples]
y_train_small = y_train[:n_samples]
print(f" Using {n_samples} samples ({sample_ratio*100:.0f}%) for hyperparameter tuning")

# ===============================
# Step 3. Define the model and hyperparameter space
# ===============================
param_grid = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [2, 5, 10, 20],
    'criterion': ['gini', 'entropy']
}

clf = DecisionTreeClassifier(random_state=42)

### Fully connected neural network

In [ ]:
# ===============================================================
#  4. Custom MLP 
# ===============================================================
import os, time, json, itertools, random, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
warnings.filterwarnings("ignore")

# --- Drawing Style ---
plt.rcParams['font.family'] = 'Times New Roman'
deep_blue, deep_red = "#003366", "#990000"

BASE = "Assignment2Data/"
os.makedirs(BASE, exist_ok=True)

# MLP
class CustomMLP:
    def __init__(self, layer_sizes, learning_rate=1e-3, l2_lambda=1e-4,
                 activation="relu", epochs=40, batch_size=128, early_stop_patience=6, seed=0):
        np.random.seed(seed)
        self.layer_sizes = layer_sizes
        self.learning_rate = float(learning_rate)
        self.l2_lambda = float(l2_lambda)
        self.activation = activation
        self.epochs = int(epochs)
        self.batch_size = int(batch_size)
        self.early_stop_patience = int(early_stop_patience)

        #Initialization
        self.W = [np.random.randn(n_in, n_out) * np.sqrt(2.0/n_in)
                  for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:])]
        self.b = [np.zeros((1, n_out)) for n_out in layer_sizes[1:]]

    # Activation function and derivative 
    def _act(self, Z):
        if self.activation == "relu":
            return np.maximum(0, Z)
        elif self.activation == "tanh":
            return np.tanh(Z)
        elif self.activation == "sigmoid":
            return 1.0/(1.0 + np.exp(-Z))

    def _act_grad(self, A):
        if self.activation == "relu":
            return (A > 0).astype(A.dtype)
        elif self.activation == "tanh":
            return 1.0 - A*A
        elif self.activation == "sigmoid":
            return A*(1.0 - A)

    def _softmax(self, Z):
        Z = Z - np.max(Z, axis=1, keepdims=True)
        e = np.exp(Z)
        return e / np.sum(e, axis=1, keepdims=True)

    def _forward(self, X):
        A = X
        As = [A]; Zs = []
        for W, b in zip(self.W[:-1], self.b[:-1]):
            Z = A @ W + b
            A = self._act(Z)
            Zs.append(Z); As.append(A)
        Z = A @ self.W[-1] + self.b[-1]
        A = self._softmax(Z)
        Zs.append(Z); As.append(A)
        return Zs, As

    def _backward(self, X, Y, Zs, As):
        m = X.shape[0]
        dW_list, db_list = [], []
        dZ = As[-1] - Y  # softmax + CE
        for i in reversed(range(len(self.W))):
            A_prev = As[i]
            dW = (A_prev.T @ dZ) / m + (self.l2_lambda/m) * self.W[i]
            db = np.sum(dZ, axis=0, keepdims=True) / m
            dW_list.insert(0, dW); db_list.insert(0, db)
            if i != 0:
                dA_prev = dZ @ self.W[i].T
                dZ = dA_prev * self._act_grad(As[i])
        return dW_list, db_list

    def _update(self, dW_list, db_list):
        for i in range(len(self.W)):
            self.W[i] -= self.learning_rate * dW_list[i]
            self.b[i] -= self.learning_rate * db_list[i]

    def fit(self, X_tr, Y_tr, X_val=None, Y_val=None, verbose=False):
        best_loss = np.inf
        best_W, best_b = None, None
        wait = 0

        for ep in range(self.epochs):
            # mini-batch SGD
            idx = np.random.permutation(X_tr.shape[0])
            X_tr, Y_tr = X_tr[idx], Y_tr[idx]
            for s in range(0, X_tr.shape[0], self.batch_size):
                Xb, Yb = X_tr[s:s+self.batch_size], Y_tr[s:s+self.batch_size]
                Zs, As = self._forward(Xb)
                dW, db = self._backward(Xb, Yb, Zs, As)
                self._update(dW, db)

            # Early stopping on the validation set
            if X_val is not None:
                P = self.predict_proba(X_val)
                val_loss = -np.mean(np.sum(Y_val*np.log(P+1e-9), axis=1))
                if verbose:
                    print(f"epoch {ep+1}/{self.epochs}  val_loss={val_loss:.4f}")
                if val_loss < best_loss:
                    best_loss = val_loss
                    best_W = [w.copy() for w in self.W]
                    best_b = [b.copy() for b in self.b]
                    wait = 0
                else:
                    wait += 1
                    if wait >= self.early_stop_patience:
                        if verbose: print("Early stopping.")
                        break

        if best_W is not None:
            self.W, self.b = best_W, best_b

    def predict_proba(self, X):
        _, As = self._forward(X)
        return As[-1]

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

### Convolutional neural network

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

In [ ]:
augment = keras.Sequential([
    layers.RandomCrop(32, 32),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
])

In [ ]:
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_test  = np.load("X_test.npy")
y_test  = np.load("y_test.npy")

In [ ]:
X_train = X_train.astype("float32")
X_test  = X_test.astype("float32")
if X_train.max() > 1.0:
    X_train /= 255.0
    X_test  /= 255.0

normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train)  # compute statistics using only the training set

input_shape = (32, 32, 3)                     # key
inputs = keras.Input(shape=input_shape)

x = inputs
x = augment(x)                                # RandomCrop/Flip/Rotation, etc.
x = normalizer(x)

In [ ]:
num_classes = int(np.max(y_train)) + 1
input_shape = X_train.shape[1:]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

AUTOTUNE = tf.data.AUTOTUNE
batch_size = 64

ds_tr  = tf.data.Dataset.from_tensor_slices((X_tr,  y_tr)).shuffle(10000).batch(batch_size).prefetch(AUTOTUNE)
ds_val = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size).prefetch(AUTOTUNE)
ds_te  = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(128).prefetch(AUTOTUNE)


for xb, yb in ds_tr.take(1):
    print("train batch:", xb.shape, yb.shape)

for xb, yb in ds_val.take(1):
    print("val   batch:", xb.shape, yb.shape)


print("train cardinality:", tf.data.experimental.cardinality(ds_tr).numpy())
print("val   cardinality:", tf.data.experimental.cardinality(ds_val).numpy())
print("test  cardinality:", tf.data.experimental.cardinality(ds_te).numpy())

In [ ]:
def build_cnn(input_shape, num_classes, base=32, dropout=0.3, use_bn=True):
    inputs = keras.Input(shape=input_shape)

    # Preprocessing and augmentation
    x = augment(inputs)          # Enabled during training, automatically disabled during evaluation
    x = normalizer(x)            # Standardize using statistics computed from the training set
    # Block 1
    x = layers.Conv2D(base, 3, padding="same", use_bias=not use_bn)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(base, 3, padding="same", use_bias=not use_bn)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(dropout)(x)

    # Block 2
    x = layers.Conv2D(base*2, 3, padding="same", use_bias=not use_bn)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(base*2, 3, padding="same", use_bias=not use_bn)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(dropout)(x)

    # Block 3
    x = layers.Conv2D(base*4, 3, padding="same", use_bias=not use_bn)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(base*4, 3, padding="same", use_bias=not use_bn)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(dropout)(x)

    # Lightweight head: GAP is more stable than Flatten
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, use_bias=not use_bn)(x)
    if use_bn: x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="CIFAR10_ScratchCNN")

model = build_cnn(input_shape, num_classes, base=32, dropout=0.3, use_bn=True)
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=3, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("cnn_best.keras", monitor="val_accuracy", save_best_only=True),
]

history = model.fit(
    ds_tr,
    validation_data=ds_val,
    epochs=30,
    callbacks=callbacks,
    verbose=2,
)

In [ ]:
# Evaluate model performance on the test set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)

print(f"✅ Test accuracy: {test_acc:.4f}")
print(f"✅ Test loss: {test_loss:.4f}")

## 3. Hyperparameter tuning

### Algorithm of choice from first six weeks of course

In [ ]:
X_train = np.load('X_train_preprocessed.npy').astype(np.float16)
X_test = np.load('X_test_preprocessed.npy').astype(np.float16)
y_train = np.load('y_train_preprocessed.npy')
y_test = np.load('y_test_preprocessed.npy')

In [ ]:
# ===============================
# Step 4. Random Search + Cross Validation (Save Memory)
# ===============================
print("\n Running RandomizedSearchCV...")
search = RandomizedSearchCV(
    estimator=clf,
    param_distributions=param_grid,
    n_iter=6,       # Randomly select 6 parameter combinations
    cv=3,           # 3-fold cross validation
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42,
    return_train_score=False
)

start_time = time.time()
search.fit(X_train_small, y_train_small)
elapsed_time = time.time() - start_time

print(f"\n Search completed in {elapsed_time:.2f} seconds")
print("Best parameters:", search.best_params_)
print("Best CV accuracy:", search.best_score_)

# ===============================
# Step 5. Evaluate the best model on the test set
# ===============================
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
print("\n Model Evaluation on Test Set")
print("Test accuracy:", test_acc)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ===============================
# Step 6. Record and visualize hyperparameter search results
# ===============================
results = pd.DataFrame(search.cv_results_)

# Add total run time column (shared by all parameters)
results['total_runtime_sec'] = elapsed_time
# Average search time per group
results['avg_fit_time_sec'] = results['mean_fit_time']

# --- Visualization 1：max_depth vs mean_test_score ---
plt.figure(figsize=(8, 6))
for criterion in results['param_criterion'].unique():
    subset = results[results['param_criterion'] == criterion]
    plt.plot(subset['param_max_depth'], subset['mean_test_score'], marker='o', label=f"{criterion}")

plt.title("Decision Tree Hyperparameter Search (RandomizedSearchCV)")
plt.xlabel("max_depth")
plt.ylabel("Cross-Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# --- Visualization 2: Average fitting time vs accuracy ---
plt.figure(figsize=(8, 6))
plt.scatter(results['avg_fit_time_sec'], results['mean_test_score'], c='blue', s=60)
plt.title("Fit Time vs CV Accuracy")
plt.xlabel("Average Fit Time (seconds)")
plt.ylabel("Mean CV Accuracy")
plt.grid(True)
plt.tight_layout()
plt.show()

print("All steps completed successfully!")

### Fully connected neural network

In [ ]:
#Normalization
X_train_f = X_train.astype("float32") / 255.0
X_test_f = X_test.astype("float32") / 255.0

# Flatten
X_train_vec = X_train_f.reshape(X_train_f.shape[0], -1)
X_test_vec = X_test_f.reshape(X_test_f.shape[0], -1)

# One-hot coding
y_train = y_train.reshape(-1)
y_test = y_test.reshape(-1)
try:
    enc = OneHotEncoder(sparse_output=False)
except TypeError:
    enc = OneHotEncoder(sparse=False)

y_train_oh = enc.fit_transform(y_train[:, None])
y_test_oh = enc.transform(y_test[:, None])

# Training/validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_vec, y_train_oh, test_size=0.2, random_state=0, stratify=y_train
)

In [ ]:
# ===============================================================
# 5. MLP Hyperparameter Tuning  (activation included)
#  Two-stage random search using CustomMLP (no keras.models)
#  Saves: Assignment2Data/mlp_tuning_custom_refined.csv
#         Assignment2Data/mlp_best_custom_refined.json
#         Assignment2Data/mlp_refined_top_scores.png
#         Assignment2Data/mlp_refined_top_times.png
# ===============================================================

import os, time, json, itertools, random, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
warnings.filterwarnings("ignore")

# --- Plot style ---
plt.rcParams['font.family'] = 'Times New Roman'
deep_blue, deep_red = "#003366", "#990000"
BASE = "Assignment2Data/"
os.makedirs(BASE, exist_ok=True)

# ===============================================================
# (1) Use a smaller subset for quick parameter tuning
# ===============================================================
def make_subset(X, Y, n):
    n = min(n, X.shape[0])
    idx = np.random.choice(X.shape[0], n, replace=False)
    return X[idx].astype(np.float32), Y[idx].astype(np.float32)

USE_PILOT = True
N_TR_PILOT = 12000
N_VAL_PILOT = 3000

X_tr_tune, y_tr_tune = (make_subset(X_tr, y_tr, N_TR_PILOT) if USE_PILOT else (X_tr, y_tr))
X_val_tune, y_val_tune = (make_subset(X_val, y_val, N_VAL_PILOT) if USE_PILOT else (X_val, y_val))

print("Tuning data:",
      f"train={X_tr_tune.shape, y_tr_tune.shape},",
      f"val={X_val_tune.shape, y_val_tune.shape}")

# ===============================================================
# 2. Define a random search function (robust + stratified by activation)
# ===============================================================
def run_random_search(param_grid, tag, n_iter=None,
                      epochs=12, patience=4,
                      balance_on: str = "activation"):
    """
    Randomly search for hyperparameters in param_grid (dict of lists).
    - Robust to key order; unpacks by param name.
    - If balance_on is in param_grid, do stratified sampling over that key.
    """
    # Build all combos as dicts
    keys = list(param_grid.keys())
    values = [param_grid[k] for k in keys]
    all_combos = [dict(zip(keys, prod)) for prod in itertools.product(*values)]

    # Helper: shuffle with fixed seed for reproducibility
    rng = random.Random(0)

    # Stratified sampling on a specific key (e.g., 'activation')
    if n_iter is not None and n_iter < len(all_combos) and (balance_on in param_grid):
        # group by balance key
        groups = {}
        for c in all_combos:
            groups.setdefault(c[balance_on], []).append(c)
        # shuffle each group
        for g in groups.values():
            rng.shuffle(g)
        # allocate roughly equal samples per group
        k = len(groups)
        base = n_iter // k
        rem = n_iter - base * k
        search_space = []
        # take base from each group
        for act, g in groups.items():
            search_space.extend(g[:base])
        # distribute remainder
        # flatten remaining pool
        leftovers = []
        for g in groups.values():
            leftovers.extend(g[base:])
        rng.shuffle(leftovers)
        search_space.extend(leftovers[:rem])
    else:
        rng.shuffle(all_combos)
        if n_iter is None or n_iter > len(all_combos):
            n_iter = len(all_combos)
        search_space = all_combos[:n_iter]

    records = []
    t0_all = time.time()

    for i, cfg in enumerate(search_space, 1):
        hls = cfg["hidden_layer_sizes"]
        lr  = float(cfg["learning_rate"])
        l2  = float(cfg["l2_lambda"])
        act = cfg["activation"]
        bs  = int(cfg["batch_size"])

        print(f"\n[{tag}] Combo {i}/{len(search_space)}:"
              f" hls={hls}, lr={lr}, l2={l2}, act={act}, batch={bs}")

        model = CustomMLP(
            layer_sizes=[3072, *hls, 10],
            learning_rate=lr, l2_lambda=l2, activation=act,
            epochs=epochs, batch_size=bs, early_stop_patience=patience, seed=0
        )

        # --- Timing and Training ---
        t0 = time.time()
        model.fit(X_tr_tune, y_tr_tune, X_val_tune, y_val_tune, verbose=True)
        elapsed = time.time() - t0

        # --- Evaluate the validation set ---
        val_pred = model.predict(X_val_tune)
        val_acc = accuracy_score(np.argmax(y_val_tune, axis=1), val_pred)
        print(f"→ {tag} val_acc={val_acc:.4f}, time={elapsed:.1f}s")

        rec = {
            "stage": tag,
            "hidden_layer_sizes": str(hls),
            "learning_rate": lr,
            "l2_lambda": l2,
            "activation": act,
            "batch_size": bs,
            "val_acc": float(val_acc),
            "fit_time_s": float(elapsed),
        }
        records.append(rec)

    print(f"\n[{tag}] total time: {time.time() - t0_all:.1f}s")
    return pd.DataFrame(records)

# ===============================================================
# (3) Stage 1: Local refinement (now includes activation search)
# ===============================================================
param_grid_stage1 = {
    "hidden_layer_sizes": [[384,192], [384,224], [384,160], [416,208], [320,160]],
    "learning_rate":      [0.02, 0.015, 0.01, 0.007, 0.005],
    "l2_lambda":          [3e-7, 1e-6, 3e-6, 1e-5],
    "activation":         ["tanh", "relu", "sigmoid"],  # newly added activations for search
    "batch_size":         [64, 128],
}

# Randomly sample hyperparameters including three activations, with stratified sampling by activation
df_s1 = run_random_search(param_grid_stage1, tag="stage1",
                          n_iter=30, epochs=12, patience=4,
                          balance_on="activation")

# --- Stage 1: pick best numeric hyperparams and top activations ---
best_s1_row = df_s1.sort_values("val_acc", ascending=False).iloc[0]
best_lr = float(best_s1_row["learning_rate"])
best_l2 = float(best_s1_row["l2_lambda"])

# You can also select top-1 or top-2 activations based on average validation accuracy for Stage 2
act_stats = df_s1.groupby("activation")["val_acc"].mean().sort_values(ascending=False)
top_activations = list(act_stats.index[:2])  # take the best two by mean performance
print("\n[stage1] best row:",
      f"hls={best_s1_row['hidden_layer_sizes']}, lr={best_lr}, l2={best_l2},",
      f"act={best_s1_row['activation']}, batch={int(best_s1_row['batch_size'])},",
      f"acc={best_s1_row['val_acc']:.4f}")
print("[stage1] activation mean val_acc ranking:")
print(act_stats)

# ===============================================================
# (4) Stage 2: Structural exploration near best lr/L2 and top activations
# ===============================================================
param_grid_stage2 = {
    "hidden_layer_sizes": [[384,192,96], [320,160,80], [512], [640]],
    "learning_rate":      [best_lr, max(best_lr*0.7, 1e-4)],
    "l2_lambda":          [best_l2, max(best_l2*3.0, 1e-7)],
    "activation":         top_activations,   # use top 1~2 activations from Stage 1
    "batch_size":         [64],
}

df_s2 = run_random_search(param_grid_stage2, tag="stage2",
                          n_iter=None, epochs=14, patience=5,
                          balance_on="activation")

# ===============================================================
# (5) Merge results + save the best model
# ===============================================================
df_all = pd.concat([df_s1, df_s2], ignore_index=True).sort_values("val_acc", ascending=False)
save_csv = os.path.join(BASE, "mlp_tuning_custom_refined.csv")
df_all.to_csv(save_csv, index=False)

best_all = df_all.iloc[0].to_dict()
save_json = os.path.join(BASE, "mlp_best_custom_refined.json")
with open(save_json, "w") as f:
    json.dump(best_all, f, indent=2)

print("\nSaved results:")
print(" -", save_csv)
print(" -", save_json)
print("Best (refined):", best_all)

# ===============================================================
# (6) Visualization results  (label includes activation)
# ===============================================================
topN = min(12, len(df_all))
top = df_all.head(topN)

plt.figure(figsize=(10,4.5))
plt.bar(range(topN), top["val_acc"], color=deep_blue)
xticklabels = [f"{hls} / {act}" for hls, act in zip(top["hidden_layer_sizes"], top["activation"])]
plt.xticks(range(topN), xticklabels, rotation=50, ha="right")
plt.ylabel("Validation accuracy", color=deep_blue)
plt.title("Top configs — Custom MLP (Refined Search with Activations)", color=deep_red, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE, "mlp_refined_top_scores.png"), dpi=200)
plt.show()

plt.figure(figsize=(7.2,4.2))
plt.bar(range(topN), top["fit_time_s"], color=deep_blue)
plt.xticks(range(topN), range(1, topN+1))
plt.xlabel("Top rank", color=deep_blue)
plt.ylabel("Fit time (s)", color=deep_blue)
plt.title("Runtime of top configs — Custom MLP (Refined, Activations)", color=deep_red, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE, "mlp_refined_top_times.png"), dpi=200)
plt.show()

### Convolutional neural network

In [ ]:
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_test  = np.load("X_test.npy")
y_test  = np.load("y_test.npy")

In [ ]:
# 1 Normalize to [0,1]
X_train = X_train.astype("float32")
X_test  = X_test.astype("float32")
if X_train.max() > 1.0:
    X_train /= 255.0
    X_test  /= 255.0

# 2 Standardization — compute statistics using training set only
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train)  # calculate mean and variance only on training set

# Load the raw image (100 here is just an example)
X_train_raw = np.load('Assignment2Data/X_train.npy')

# 3 Select the 100th image for demonstration
idx = 100
original_img   = X_train_raw[idx]  # Original unnormalized image (uint8, 0~255)
normalized_img = X_train[idx]      # Normalized image (float32, 0~1)

# 4 Standardize (Normalization layer expects shape (1, 32, 32, 3))
img_norm_batched = normalizer(normalized_img[None, ...])  # explicitly add batch dimension
img_norm = img_norm_batched[0].numpy()                    # remove batch dimension -> (32, 32, 3)

In [ ]:
# ==== CNN Hyperparameter Tuning (Independent Cell) ====
# --- SAFETY SWITCH FOR MARKERS ---
RUN_TUNING = False

import os, time, json, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

# 1) Data preparation (normalizes X_train / y_train / X_test / y_test if not yet processed)
def prepare_data(Xtr, ytr, Xte, yte, test_size=0.2, batch_size=64):
    Xtr = Xtr.astype("float32"); Xte = Xte.astype("float32")
    if Xtr.max() > 1.0:
        Xtr /= 255.0; Xte /= 255.0
    X_tr, X_val, y_tr, y_val = train_test_split(
        Xtr, ytr, test_size=test_size, random_state=42, stratify=ytr
    )
    AUTOTUNE = tf.data.AUTOTUNE
    def make_ds(X, y, training):
        ds = tf.data.Dataset.from_tensor_slices((X, y))
        if training: ds = ds.shuffle(10000, seed=42, reshuffle_each_iteration=True)
        return ds.batch(batch_size).prefetch(AUTOTUNE)
    return X_tr, X_val, y_tr, y_val, make_ds

# 2) Model building function (configurable parameters)
def build_cnn(input_shape, num_classes,
              base_channels=32, dropout=0.4, use_gap=True, weight_decay=1e-4):
    reg = keras.regularizers.l2(weight_decay) if weight_decay else None
    inputs = keras.Input(shape=input_shape)
    x = inputs
    # Data augmentation (applied only during training)
    x = layers.RandomFlip("horizontal")(x)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)
    for filters in [base_channels, base_channels*2, base_channels*4]:
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_regularizer=reg)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPool2D()(x)
    x = layers.GlobalAveragePooling2D()(x) if use_gap else layers.Flatten()(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=reg)(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(int(np.max(y_train))+1, activation="softmax")(x)
    return keras.Model(inputs, outputs)

# 3) Single run training and evaluation
def train_and_eval(config):
    X_tr, X_val, y_tr, y_val, make_ds = prepare_data(X_train, y_train, X_test, y_test,
                                                     test_size=0.2, batch_size=config["batch_size"])
    ds_tr  = make_ds(X_tr,  y_tr,  training=True)
    ds_val = make_ds(X_val, y_val, training=False)

    # Model and optimizer
    model = build_cnn(
        input_shape=X_train.shape[1:], num_classes=int(np.max(y_train))+1,
        base_channels=config["base_channels"],
        dropout=config["dropout"],
        use_gap=config["use_gap"],
        weight_decay=config["weight_decay"]
    )
    if config["optimizer"] == "adamw":
        opt = keras.optimizers.AdamW(learning_rate=config["lr"], weight_decay=config["weight_decay"])
    else:
        opt = keras.optimizers.Adam(learning_rate=config["lr"])

    model.compile(optimizer=opt,
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy"])

    cbs = [
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True)
    ]

    t0 = time.time()
    hist = model.fit(ds_tr, validation_data=ds_val,
                     epochs=config["epochs"], verbose=2, callbacks=cbs)
    secs = time.time() - t0

    best_val = float(np.max(hist.history["val_accuracy"]))
    best_epoch = int(np.argmax(hist.history["val_accuracy"]) + 1)

    # Callbacks (learning rate schedule & early stopping)
    ds_te = tf.data.Dataset.from_tensor_slices((X_test.astype("float32")/255.0 if X_test.max()>1 else X_test,
                                                y_test)).batch(128).prefetch(tf.data.AUTOTUNE)
    test_loss, test_acc = model.evaluate(ds_te, verbose=0)

    return {
        "config": config,
        "val_acc": best_val,
        "test_acc": float(test_acc),
        "best_epoch": best_epoch,
        "time_sec": round(secs, 1)
    }

# 4) Search space (at least 3 hyperparameters)
search_space = [
    {"lr":1e-3, "batch_size":64,  "base_channels":32, "dropout":0.3, "weight_decay":0.0,  "use_gap":True,  "optimizer":"adam",  "epochs":30},
    {"lr":3e-4, "batch_size":64,  "base_channels":48, "dropout":0.4, "weight_decay":1e-4, "use_gap":True,  "optimizer":"adamw", "epochs":30},
    {"lr":1e-4, "batch_size":128, "base_channels":64, "dropout":0.5, "weight_decay":1e-4, "use_gap":False, "optimizer":"adamw", "epochs":30},
    {"lr":5e-4, "batch_size":128, "base_channels":48, "dropout":0.3, "weight_decay":5e-5, "use_gap":True,  "optimizer":"adam",  "epochs":30},
]

# 5) Run search and print results
results = [train_and_eval(cfg) for cfg in search_space]
for r in results:
    cfg = r["config"]
    print(
        f"cfg: lr={cfg['lr']}, bs={cfg['batch_size']}, base={cfg['base_channels']}, "
        f"dropout={cfg['dropout']}, wd={cfg['weight_decay']}, gap={cfg['use_gap']}, opt={cfg['optimizer']}  "
        f"| val={r['val_acc']:.4f} test={r['test_acc']:.4f} "
        f"@{r['best_epoch']}ep  {r['time_sec']}s"
    )

# 6) Select the best configuration (for reporting only, not used in final section)
best = max(results, key=lambda x: x["val_acc"])
print("\nBEST by val_acc:", json.dumps(best, indent=2))

## 4. Final models
In this section, please ensure to include cells to train each model with its best hyperparmater combination independently of the hyperparameter tuning cells, i.e. don't rely on the hyperparameter tuning cells having been run.

### Algorithm of choice from first six weeks of course

In [ ]:
X_train = np.load('X_train_preprocessed.npy').astype(np.float16)
X_test = np.load('X_test_preprocessed.npy').astype(np.float16)
y_train = np.load('y_train_preprocessed.npy')
y_test = np.load('y_test_preprocessed.npy')

In [ ]:
# ===============================================================
# Decision Tree Final Model Training and Evaluation 
# ===============================================================
import json, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

plt.rcParams['font.family'] = 'Times New Roman'
deep_blue, deep_red = "#003366", "#990000"

# ---------- (0) Load data if not already loaded ----------
try:
    X_train, X_test, y_train, y_test
except NameError:
    # If the data is not loaded, reload it
    print("Loading preprocessed data...")
    X_train = np.load('X_train_preprocessed.npy').astype(np.float32)
    X_test = np.load('X_test_preprocessed.npy').astype(np.float32)
    y_train = np.load('y_train_preprocessed.npy')
    y_test = np.load('y_test_preprocessed.npy')

print(f"Data shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

# ---------- (1) Define the best parameters ----------
# Based on your tuning results
best_params = {
    "max_depth": 20,
    "min_samples_split": 20, 
    "criterion": "gini",
    "random_state": 42
}

print("Best parameters for final model:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

# ---------- (2) Define the final model ----------
final_model = DecisionTreeClassifier(
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    criterion=best_params["criterion"],
    random_state=best_params["random_state"]
)

# ---------- (3) Training ----------
print("\nTraining final Decision Tree model...")
t0 = time.time()
final_model.fit(X_train, y_train)
train_time = time.time() - t0
print(f"Training completed in {train_time:.2f} seconds")

# ---------- (4) Test set evaluation ----------
print("\nEvaluating on test set...")
y_pred = final_model.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
print(f"\n Final Test Accuracy: {test_acc:.4f}")
print("\n Detailed Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

# ---------- (5) Confusion Matrix ----------
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", 
            xticklabels=range(10), yticklabels=range(10))
plt.title("Confusion Matrix — Final Decision Tree", color=deep_red, fontweight="bold", fontsize=13)
plt.xlabel("Predicted label", color=deep_blue)
plt.ylabel("True label", color=deep_blue)
plt.tight_layout()
plt.savefig("decision_tree_final_confusion_matrix.png", dpi=200)
plt.show()

# ---------- (6) Feature Importance (if applicable) ----------
if hasattr(final_model, 'feature_importances_'):
    plt.figure(figsize=(12, 6))
    feature_importance = final_model.feature_importances_
    # Take the top 30 most important features for visualization
    top_k = min(30, len(feature_importance))
    indices = np.argsort(feature_importance)[-top_k:]
    
    plt.barh(range(top_k), feature_importance[indices])
    plt.yticks(range(top_k), [f'Feature {i}' for i in indices])
    plt.title(f"Top {top_k} Feature Importances — Decision Tree", color=deep_red, fontweight="bold")
    plt.xlabel("Feature Importance", color=deep_blue)
    plt.tight_layout()
    plt.savefig("decision_tree_feature_importance.png", dpi=200)
    plt.show()

# ---------- (7) Save the results ----------
results = {
    'test_acc': test_acc,
    'train_time': train_time,
    'best_params': best_params,
    'predictions': y_pred,
    'true_labels': y_test
}

# Save as npz file
np.savez_compressed(
    "decision_tree_final_results.npz",
    **results
)

# Save the model
import joblib
joblib.dump(final_model, 'decision_tree_final_model.pkl')

# Save text results
with open("decision_tree_final_results.txt", "w") as f:
    f.write("Final Decision Tree Model Results\n")
    f.write("=" * 50 + "\n")
    f.write(f"Test Accuracy: {test_acc:.4f}\n")
    f.write(f"Training Time: {train_time:.2f} seconds\n")
    f.write(f"Best Parameters:\n")
    for k, v in best_params.items():
        f.write(f"  {k}: {v}\n")
    f.write(f"Training samples: {X_train.shape[0]}\n")
    f.write(f"Test samples: {X_test.shape[0]}\n")

print("\n Results saved to:")
print("   - decision_tree_final_results.npz")
print("   - decision_tree_final_results.txt") 
print("   - decision_tree_final_model.pkl")
print("   - decision_tree_final_confusion_matrix.png")
if hasattr(final_model, 'feature_importances_'):
    print("   - decision_tree_feature_importance.png")

### Fully connected neural network

In [ ]:
# ===============================================================
# 6. Final Model Training and Evaluation 
# ===============================================================
import json, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

plt.rcParams['font.family'] = 'Times New Roman'
deep_blue, deep_red = "#003366", "#990000"

# ---------- (1) Read the optimal parameters ----------
best_path = "Assignment2Data/mlp_best_custom_refined.json"
with open(best_path, "r") as f:
    best_cfg = json.load(f)

print("Loaded best refined config:")
for k, v in best_cfg.items():
    print(f"  {k}: {v}")

# ---------- (2) Combine training + validation set ----------
X_full = np.vstack([X_tr, X_val])
y_full = np.vstack([y_tr, y_val])

# ---------- (3) Define the final model ----------
final_model = CustomMLP(
    layer_sizes=[3072, *eval(best_cfg["hidden_layer_sizes"]), 10],
    learning_rate=float(best_cfg["learning_rate"]),
    l2_lambda=float(best_cfg["l2_lambda"]),
    activation=best_cfg["activation"],
    batch_size=int(best_cfg["batch_size"]),
    epochs=18,               # train longer for better convergence
    early_stop_patience=5,   # increase tolerance
    seed=0
)

# ---------- (4) Training ----------
print("\nTraining final model on (train + val) ...")
t0 = time.time()
final_model.fit(X_full, y_full, verbose=True)
train_time = time.time() - t0
print(f"\n Training completed in {train_time/60:.2f} minutes")

# ---------- (5) Test set evaluation (!!! fixed: use X_test_vec !!!) ----------
print("\nEvaluating on test set ...")
expected_in = final_model.layer_sizes[0]
assert X_test_vec.shape[1] == expected_in, " X_test_vec dimension mismatch (should be flattened 3072)"
y_pred = final_model.predict(X_test_vec)
y_true = np.argmax(y_test_oh, axis=1)

test_acc = accuracy_score(y_true, y_pred)
print(f"\n Final Test Accuracy: {test_acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

# ---------- (6) Confusion Matrix ----------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, cmap="RdBu_r", annot=False)
plt.title("Confusion Matrix — Final MLP (Fixed)", color=deep_red, fontweight="bold", fontsize=13)
plt.xlabel("Predicted label", color=deep_blue)
plt.ylabel("True label", color=deep_blue)
plt.tight_layout()
plt.savefig("Assignment2Data/mlp_final_confusion_matrix_fixed.png", dpi=200)
plt.show()

# ---------- (7) Save the result ----------
np.savez_compressed(
    "Assignment2Data/mlp_final_results_fixed.npz",
    test_acc=test_acc,
    conf_mat=cm,
    train_time=train_time,
    best_cfg=json.dumps(best_cfg)
)
print("\nResults saved to Assignment2Data/mlp_final_results_fixed.npz")

### Convolutional neural network

In [ ]:
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_test  = np.load("X_test.npy")
y_test  = np.load("y_test.npy")

In [ ]:
# 1 Normalize to [0,1]
X_train = X_train.astype("float32")
X_test  = X_test.astype("float32")
if X_train.max() > 1.0:
    X_train /= 255.0
    X_test  /= 255.0

# 2 Standardization — compute statistics using training set only
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train)  # calculate mean and variance only on training set

# Load the raw image (100 here is just an example)
X_train_raw = np.load('Assignment2Data/X_train.npy')

# 3 Select the 100th image for demonstration
idx = 100
original_img   = X_train_raw[idx]  # Original unnormalized image (uint8, 0~255)
normalized_img = X_train[idx]      # Normalized image (float32, 0~1)

# 4 Standardize (Normalization layer expects shape (1, 32, 32, 3))
img_norm_batched = normalizer(normalized_img[None, ...])  # explicitly add batch dimension
img_norm = img_norm_batched[0].numpy()                    # remove batch dimension -> (32, 32, 3)

In [ ]:
# ==== Final CNN (independent; local .npy; no dependency on tuning cells) ====
import os, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

# 1) Local relative path (assume .ipynb is in the Assignment2Data folder)
data_dir = os.path.join(os.getcwd(), "Assignment2Data")

def _load(name):
    p = os.path.join(data_dir, name)
    assert os.path.exists(p), f"Missing file: {p}"
    return np.load(p)

X_train = _load("X_train.npy").astype("float32")
y_train = _load("y_train.npy").reshape(-1)
X_test  = _load("X_test.npy").astype("float32")
y_test  = _load("y_test.npy").reshape(-1)
if X_train.max() > 1.0:
    X_train /= 255.0; X_test /= 255.0

num_classes = int(np.max(y_train)) + 1
input_shape = X_train.shape[1:]

# 2) Hard-coded best hyperparameters (from your tuning logs)
BEST_CNN = {"lr":5e-4,"batch":128,"base":48,"dropout":0.3,"wd":5e-05,"use_gap":True,"optimizer":"adam","epochs":30}

# 3) Data pipeline
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
AUTOTUNE = tf.data.AUTOTUNE
def make_ds(X,y,b,train=False):
    ds=tf.data.Dataset.from_tensor_slices((X,y))
    if train:
        ds=ds.shuffle(10_000,seed=42,reshuffle_each_iteration=True)
        ds=ds.map(lambda x,y:(tf.image.random_flip_left_right(x),y), num_parallel_calls=AUTOTUNE)
    return ds.batch(b).prefetch(AUTOTUNE)

ds_tr  = make_ds(X_tr,y_tr,BEST_CNN["batch"],True)
ds_val = make_ds(X_val,y_val,BEST_CNN["batch"],False)
ds_te  = make_ds(X_test,y_test,256,False)

# 4) Model structure (consistent with tuning phase)
reg = keras.regularizers.l2(BEST_CNN["wd"])
def block(x,c):
    x=layers.Conv2D(c,3,padding='same',use_bias=False,kernel_regularizer=reg)(x); x=layers.BatchNormalization()(x); x=layers.ReLU()(x)
    x=layers.Conv2D(c,3,padding='same',use_bias=False,kernel_regularizer=reg)(x); x=layers.BatchNormalization()(x); x=layers.ReLU()(x)
    x=layers.MaxPooling2D(2)(x); x=layers.Dropout(BEST_CNN["dropout"])(x); return x

inp=keras.Input(shape=input_shape)
x=layers.Conv2D(BEST_CNN["base"],3,padding='same',use_bias=False,kernel_regularizer=reg)(inp); x=layers.BatchNormalization()(x); x=layers.ReLU()(x)
x=block(x,BEST_CNN["base"]); 
x=block(x,BEST_CNN["base"]*2); 
x=block(x,BEST_CNN["base"]*3)
x=layers.GlobalAveragePooling2D()(x) if BEST_CNN["use_gap"] else layers.Flatten()(x)
x=layers.Dropout(BEST_CNN["dropout"])(x)
out=layers.Dense(num_classes,activation='softmax')(x)
model=keras.Model(inp,out)

opt = (keras.optimizers.Adam if BEST_CNN["optimizer"]=="adam" else keras.optimizers.AdamW)(learning_rate=BEST_CNN["lr"])
model.compile(opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cbs = [
    keras.callbacks.ModelCheckpoint("cnn_final_best.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=7, restore_best_weights=True),
]
hist = model.fit(ds_tr, validation_data=ds_val, epochs=BEST_CNN["epochs"], verbose=2, callbacks=cbs)

test_loss, test_acc = model.evaluate(ds_te, verbose=0)
print(f"✅ Final CNN — best val acc: {max(hist.history['val_accuracy']):.4f} | test acc: {test_acc:.4f}")

In [ ]:
from tensorflow import keras
best = keras.models.load_model("cnn_final_best.keras")
best.evaluate(ds_te, verbose=2)